# Class 1 简明版：同一句例子，走通三个步骤

文字变数字 → Attention 混合信息 → 下一 token。下面代码与 `code/lesson.py` 相同；无需 PyTorch、NumPy 或网络。保留的输出来自实际执行。

## 准备：只用 Python 标准库

先定义一个短函数 softmax：分数变成非负、总和为 1 的比例。减去最大值用于数值稳定。本课参数人工设定，展示一次计算；完整训练实现放在 optional/。

In [1]:
import math


def softmax(scores):
    exp_scores = []
    for score in scores:
        exp_scores.append(math.exp(score - max(scores)))
    total = sum(exp_scores)
    return [value / total for value in exp_scores]

## ① 文字变数字

输入固定为“我 喜欢 喝”。先按空格切分，再查 ID，再查二维 embedding 表。预测：最后一个向量是什么？本课 ID 仅是编号，数字大小没有语义。

In [2]:
text = "我 喜欢 喝"
tokens = text.split()
vocab = {"我": 0, "喜欢": 1, "喝": 2, "酒": 3, "牛奶": 4}
token_ids = [vocab[token] for token in tokens]
embedding_table = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.5, 1.0], [0.8, 0.2]]
vectors = [embedding_table[token_id] for token_id in token_ids]
print("1. token：", tokens)
print("   ID：", token_ids)
print("   向量：", vectors)

1. token： ['我', '喜欢', '喝']
   ID： [0, 1, 2]
   向量： [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]


## ② 从最后一个位置提出查询

本例 Q/K/V 采用恒等投影，数字相同但角色不同。Q/K 用于确定权重，V 用于混合。实际模型一般分别学习三个线性投影。只算“喝”这个位置的一行。

In [3]:
query = vectors[-1]
keys = vectors
values = vectors
print("2. 当前查询 Q：", query)

2. 当前查询 Q： [1.0, 1.0]


## ② 算匹配分数与权重

逐项相乘再相加是点积，除以 √2 控制分数尺度。运行前先算三个点积：1、1、2。Softmax 后权重约为 [0.248,0.248,0.503]。显示值的和可能有四舍五入差异。

In [4]:
scores = []
for key in keys:
    dot = query[0] * key[0] + query[1] * key[1]
    scores.append(dot / math.sqrt(2))
weights = softmax(scores)
print("3. 匹配分数：", [round(score, 3) for score in scores])
print("   混合权重：", [round(weight, 3) for weight in weights])

3. 匹配分数： [0.707, 0.707, 1.414]
   混合权重： [0.248, 0.248, 0.503]


## ② 按权重混合信息

分别计算两个维度的加权和。预期输出约 [0.752,0.752]。它是向量，尚未选出 token。

In [5]:
mixed = [0.0, 0.0]
for i in range(len(tokens)):
    mixed[0] += weights[i] * values[i][0]
    mixed[1] += weights[i] * values[i][1]
print("4. Attention 输出：", [round(value, 3) for value in mixed])

4. Attention 输出： [0.752, 0.752]


## ③ 候选分数变概率，再选择

本例用人工系数把混合向量映射到两个候选的分数，Softmax 得到约 [0.680,0.320]。真实模型会输出整个词表的分数，且在完整 Block 后做输出映射。本例只演示一次选择，无训练与生成循环。

In [6]:
candidates = ["酒", "牛奶"]
logits = [2 * mixed[0], mixed[1]]  # 人工输出投影，未训练；只用两个候选演示
probabilities = softmax(logits)
for i in range(len(candidates)):
    print("5. 候选：", candidates[i], round(probabilities[i], 3))
next_token = candidates[probabilities.index(max(probabilities))]
print("   接上一个 token：", text + " " + next_token)

5. 候选： 酒 0.68
5. 候选： 牛奶 0.32
   接上一个 token： 我 喜欢 喝 酒


## 小实验：只改 V，先猜结果

保持 Q/K 和权重不变，仅把第三个 V 改成 [3,1]。新输出约为 [1.759,0.752]。这里新建一个列表，避免同时修改共享的 K。

In [7]:
changed_values = [[1, 0], [0, 1], [3, 1]]
changed_mixed = [0.0, 0.0]
for i in range(len(tokens)):
    changed_mixed[0] += weights[i] * changed_values[i][0]
    changed_mixed[1] += weights[i] * changed_values[i][1]
print("权重仍是：", [round(w, 3) for w in weights])
print("新输出：", [round(v, 3) for v in changed_mixed])

权重仍是： [0.248, 0.248, 0.503]
新输出： [1.759, 0.752]


## 因果遮罩：只看第一个位置

主例子中的最后一个位置可以读三个已知输入。如果换成第一个位置，未来位置必须遮住；在 Softmax 前设成负无穷，权重变成 0。

In [8]:
first_position_scores = [0.707, -math.inf, -math.inf]
first_position_weights = softmax(first_position_scores)
print("第一个位置的权重：", first_position_weights)

第一个位置的权重： [1.0, 0.0, 0.0]


## 把代码放回完整架构

真实 decoder-only 模型会加入位置信息，并使用含 Attention、FFN、残差和归一化的 Block；可以叠多层。本课脚本省略这些模块，不能直接称为完整 Transformer。训练会学习参数，生成不更新参数。

检查：ID 和向量有什么区别？Attention 输出为何还不能直接当作 token？两次 Softmax 的对象有什么不同？完整训练另见 [optional/](optional/README.md)。